# Hyperparameters for Black-Scholes

In [51]:
# Standard library imports
import os
import time
from pathlib import Path

# Data manipulation and mathematics
import numpy as np
import pandas as pd
from scipy.stats import norm

# Visualization
import matplotlib.pyplot as plt
from matplotlib import cm

# Deep learning framework (PyTorch)
import torch
import torch.nn as nn
import torch.optim as optim

# Sweep
import itertools

### Colab Setup

In [52]:
try:
    import google.colab
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if IN_COLAB:
    repo_url = "https://github.com/egil10/fys5429.git"
    repo_dir = "/content/fys5429"

    if not os.path.exists(repo_dir):
        !git clone {repo_url} {repo_dir}
    else:
        !git -C {repo_dir} pull

    os.chdir(os.path.join(repo_dir, "code", "notebooks"))

print(f"Working directory: {os.getcwd()}")

Already up to date.
Working directory: /content/fys5429/code/notebooks


### Paths

In [ ]:
# Pathways 
data_path = Path("..") / "data" / "generated" / "bs_collocation.parquet"
out_dir = Path("..") / "plots" / "eda"
out_dir.mkdir(parents=True, exist_ok=True)

### Global parameters

In [54]:
# Answer to the universe and everything
torch.manual_seed(42)

# Use GPU if available, otherwise CPU
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

# Option Physics (Analytical Benchmarks)
K = 100.0
r = 0.05
sigma = 0.20
T_max = 1.0
S_max = 300.0

# NN
HIDDEN_LAYERS = 3
NEURONS_PER_LAYER = 128

# Training
LEARNING_RATE = 0.0005
EPOCHS = 10000
PRINT_ROWS = 20

# PINN Loss Weights
LAMBDA_PDE = 5.0
LAMBDA_IC = 10.0
LAMBDA_BC = 5.0
LAMBDA_PDE_VALUES = [1.0, 5.0, 10.0, 20.0]
LAMBDA_IC_VALUES  = [1.0, 5.0, 10.0, 20.0]
LAMBDA_BC_FIXED   = 5.0
SWEEP_EPOCHS      = 2000

Using device: cuda


### Black-Scholes PINN Class

In [55]:
class BSPINN(nn.Module):
    def __init__(self, hidden_layers=HIDDEN_LAYERS, neurons_per_layer=NEURONS_PER_LAYER):
        super(BSPINN, self).__init__()

        # Input layer: Takes (S, tau) -> 2 dimensions
        layers = [nn.Linear(2, neurons_per_layer), nn.Softplus()]

        # Hidden layers
        # Softplus or Tanh here since ReLU has a 2nd derivative of 0,
        # which would make V_SS completely vanish in our PDE
        for _ in range(hidden_layers):
            layers.append(nn.Linear(neurons_per_layer, neurons_per_layer))
            layers.append(nn.Softplus())

        # Outer layer: Returns V (Option Price) -> 1 dimension
        layers.append(nn.Linear(neurons_per_layer, 1))

        self.net = nn.Sequential(*layers)
    
    def forward(self, S, tau):
        # concatenate S and tau into a single input tensor
        x = torch.cat([S, tau], dim=1)
        return self.net(x)


### Reading Data

In [56]:
# Loading Parquet data file into a Pandas DataFrame
df_all = pd.read_parquet(data_path)

# Helper function to extract columns and turn them into gradient-tracking Tensors
def extract_tensors(df_subset):
    S_tensor = torch.tensor(df_subset['S'].values, dtype=torch.float32).view(-1, 1).to(device).requires_grad_(True)
    tau_tensor = torch.tensor(df_subset['tau'].values, dtype=torch.float32).view(-1, 1).to(device).requires_grad_(True)
    return S_tensor, tau_tensor

# Interior points
df_interior = df_all[df_all['point_type'] == 'interior']
S_in, tau_in = extract_tensors(df_interior)

# Initial Condition points (Maturity)
df_ic = df_all[df_all['point_type'] == 'initial_condition']
S_ic, tau_ic = extract_tensors(df_ic)

# Boundary Condition points (S = 0)
df_bc = df_all[df_all['point_type'] == 'boundary_condition']
S_bc, tau_bc = extract_tensors(df_bc)

print(f"Read data from {data_path}")
print(f"Interior points: {len(S_in)}")
print(f"IC points: {len(S_ic)}")
print(f"BC points: {len(S_bc)}")

Read data from ../data/generated/bs_collocation.parquet
Interior points: 2000
IC points: 500
BC points: 500


### Training

In [57]:
# Function to train the BSPINN across sweeps
def train_pinn(lambda_pde, lambda_ic, lambda_bc, epochs, lr=0.0005, hidden_layers=3, neurons=128, verbose=False):

    model = BSPINN(hidden_layers, neurons).to(device)
    optimizer = optim.Adam(model.parameters(), lr=lr)

    grad_ones = torch.ones_like(S_in)

    history = {'epoch': [], 'pde': [], 'ic': [], 'bc': [], 'total': []}

    sample_interval = max(1, epochs // 100)

    for epoch in range(epochs):

        optimizer.zero_grad()

        # Calculating the PDE loss
        V_pred = model(S_in, tau_in)

        # Automatic differentiation for the gradients using the pre-allocated array!
        V_S = torch.autograd.grad(V_pred, S_in, grad_outputs=grad_ones, create_graph=True)[0]
        V_tau = torch.autograd.grad(V_pred, tau_in, grad_outputs=grad_ones, create_graph=True)[0]
        V_SS = torch.autograd.grad(V_S, S_in, grad_outputs=grad_ones, create_graph=True)[0]

        # Black-Scholes PDE equation
        pde_residual = V_tau - (0.5 * sigma**2 * S_in**2 * V_SS + r * S_in * V_S - r * V_pred)
        loss_pde = torch.mean(pde_residual**2)

        # Calculating the initial condition loss
        # which is the payoff at maturity
        V_ic_pred = model(S_ic, tau_ic)

        # True payoff of a Call option is maX(S-K, 0)
        V_ic_true = torch.relu(S_ic - K)
        loss_ic = torch.mean((V_ic_pred - V_ic_true)**2)

        # Calculating the boundary condition loss at S=0
        V_bc_pred = model(S_bc, tau_bc)

        # A call option is worth 0 if the stock price is 0
        loss_bc = torch.mean((V_bc_pred - 0.0)**2)

        # Total Loss and Backpropagation
        loss = lambda_pde * loss_pde + lambda_ic * loss_ic + lambda_bc * loss_bc
        loss.backward()
        optimizer.step()

        if epoch % sample_interval == 0 or epoch == epochs - 1:
          history['epoch'].append(epoch)
          history['pde'].append(loss_pde.item())
          history['ic'].append(loss_ic.item())
          history['bc'].append(loss_bc.item())
          history['total'].append(loss.item())

    return {
      'model': model,
      'history': history,
      'final_pde': loss_pde.item(),
      'final_ic': loss_ic.item(),
      'final_bc': loss_bc.item(),
      'final_total': loss.item()
    }



### Lambda Sweep

In [ ]:
# Sweeping
sweep_results = []

start_time = time.time()

for i, (lam_pde, lam_ic) in enumerate(itertools.product(LAMBDA_PDE_VALUES, LAMBDA_IC_VALUES)):
    total_runs = len(LAMBDA_PDE_VALUES) * len(LAMBDA_IC_VALUES)
    print(f"[{i+1}/{total_runs}] PDE={lam_pde:5.1f}  IC={lam_ic:5.1f}  ", end="")

    result = train_pinn(lam_pde, lam_ic, LAMBDA_BC_FIXED, SWEEP_EPOCHS)
    result['lambda_pde'] = lam_pde
    result['lambda_ic'] = lam_ic
    result['lambda_bc'] = LAMBDA_BC_FIXED

    sweep_results.append(result)
    print(f"PDE={result['final_pde']:.6f}  IC={result['final_ic']:.6f}  BC={result['final_bc']:.6f}")

elapsed = time.time() - start_time
print(f"\nSweep complete: {len(sweep_results)} runs in {int(elapsed//60)}m {int(elapsed%60)}s")

[1/16] PDE=  1.0  IC=  1.0  PDE=0.253292  IC=0.444978  BC=0.000048
[2/16] PDE=  1.0  IC=  5.0  PDE=0.997844  IC=0.443061  BC=0.000170
[3/16] PDE=  1.0  IC= 10.0  PDE=1.721690  IC=0.215763  BC=0.000774
[4/16] PDE=  1.0  IC= 20.0  PDE=4.145112  IC=0.245449  BC=0.002102
[5/16] PDE=  5.0  IC=  1.0  PDE=0.041828  IC=0.662038  BC=0.000040
[6/16] PDE=  5.0  IC=  5.0  PDE=0.190860  IC=0.448792  BC=0.000028
[7/16] PDE=  5.0  IC= 10.0  PDE=0.228317  IC=0.555805  BC=0.000013
[8/16] PDE=  5.0  IC= 20.0  PDE=0.886997  IC=0.294271  BC=0.000836
[9/16] PDE= 10.0  IC=  1.0  PDE=0.026106  IC=1.476784  BC=0.002385
[10/16] PDE= 10.0  IC=  5.0  PDE=0.117628  IC=0.917595  BC=0.001607
[11/16] PDE= 10.0  IC= 10.0  PDE=0.280053  IC=0.582973  BC=0.000169
[12/16] PDE= 10.0  IC= 20.0  PDE=0.309968  IC=0.349079  BC=0.000116
[13/16] PDE= 20.0  IC=  1.0  

### Results

In [ ]:
df_sweep = pd.DataFrame([{
    'lambda_pde': r['lambda_pde'],
    'lambda_ic': r['lambda_ic'],
    'pde_loss': r['final_pde'],
    'ic_loss': r['final_ic'],
    'bc_loss': r['final_bc'],
    'total_loss': r['final_total'],
} for r in sweep_results])

df_sweep = df_sweep.sort_values('pde_loss')
print(df_sweep.to_string(index=False))

### Heatmaps

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 8))

for ax, metric, title in zip(axes,
  ['pde_loss', 'ic_loss', 'bc_loss'],
  ['PDE Residual Loss', 'IC Loss', 'BC Loss']):
  pivot = df_sweep.pivot_table(index='lambda_ic', columns='lambda_pde', values=metric)
  log_vals = np.log10(pivot.values)
  im = ax.imshow(log_vals, cmap='viridis', aspect='auto', origin='lower')
  ax.set_xticks(range(len(LAMBDA_PDE_VALUES)))
  ax.set_xticklabels(LAMBDA_PDE_VALUES)
  ax.set_yticks(range(len(LAMBDA_IC_VALUES)))
  ax.set_yticklabels(LAMBDA_IC_VALUES)
  ax.set_xlabel('$\\lambda_{PDE}$')
  ax.set_ylabel('$\\lambda_{IC}$')
  ax.set_title(f'{title} ($\\log_{{10}}$)')
  fig.colorbar(im, ax=ax, shrink=0.8)
  for i in range(len(LAMBDA_IC_VALUES)):
      for j in range(len(LAMBDA_PDE_VALUES)):
          ax.text(j, i, f'{pivot.values[i, j]:.1e}',
                  ha='center', va='center', fontsize=7, color='white')
plt.suptitle(f'Lambda Sweep ({SWEEP_EPOCHS} epochs, $\\lambda_{{BC}}$={LAMBDA_BC_FIXED})',
           fontweight='bold', fontsize=14)

# Tight layout
plt.tight_layout()

# # Saving and showing the figure
plt.savefig(out_dir / "hyper_bs_heatmap.pdf", bbox_inches="tight")
plt.show()


### Ranked by PDE Loss

In [ ]:
df_ranked = df_sweep.sort_values('pde_loss').reset_index(drop=True)
labels = [f"PDE={r.lambda_pde:.0f}, IC={r.lambda_ic:.0f}"
          for _, r in df_ranked.iterrows()]

fig, ax = plt.subplots(figsize=(14, 5))
colors = plt.cm.viridis(np.linspace(0, 1, len(df_ranked)))
bars = ax.barh(range(len(df_ranked)), df_ranked['pde_loss'], color=colors)
ax.set_yticks(range(len(df_ranked)))
ax.set_yticklabels(labels)
ax.set_xlabel('PDE Residual Loss')
ax.set_title(f'All Lambda Combinations Ranked by PDE Loss ({SWEEP_EPOCHS} epochs)')
ax.invert_yaxis()

for i, v in enumerate(df_ranked['pde_loss']):
    ax.text(v, i, f'  {v:.2e}', va='center', fontsize=8)

plt.tight_layout()
plt.savefig(out_dir / "hyper_bs_ranked.pdf", bbox_inches="tight")
plt.show()

### Top 3 Loss Curves

In [ ]:
df_ranked = df_sweep.sort_values('pde_loss').reset_index(drop=True)
top_3_indices = df_ranked.head(3).index.tolist()

fig, ax = plt.subplots(figsize=(12, 5))

for rank, idx in enumerate(top_3_indices):
    r = sweep_results[idx] if idx < len(sweep_results) else None
    for orig in sweep_results:
        if (orig['lambda_pde'] == df_ranked.loc[idx, 'lambda_pde'] and
            orig['lambda_ic'] == df_ranked.loc[idx, 'lambda_ic']):
            r = orig
            break

    label = f"#{rank+1}: PDE={r['lambda_pde']:.0f}, IC={r['lambda_ic']:.0f}"
    ax.plot(r['history']['epoch'], r['history']['pde'], label=label, lw=1.2)

ax.set_yscale('log')
ax.set_xlabel('Epoch')
ax.set_ylabel('PDE Loss (Log Scale)')
ax.set_title('PDE Loss Convergence: Top 3 Lambda Combinations')
ax.legend()
ax.grid(True, which='both', ls='-', alpha=0.2)
plt.tight_layout()
plt.savefig(out_dir / "hyper_bs_top3_curves.pdf", bbox_inches="tight")
plt.show()

In [ ]:
if IN_COLAB:
    from google.colab import files
    for f in ["hyper_bs_heatmap.pdf", "hyper_bs_ranked.pdf", "hyper_bs_top3_curves.pdf"]:
        path = out_dir / f
        if path.exists():
            files.download(str(path))
            print(f"Downloaded: {f}")
else:
    print(f"Plots saved locally to: {out_dir.resolve()}")